# <center>Laboratorio 9: Benchmark de Carga y Modelos con Spotify 🎵</center>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

---

### Cuerpo Docente

- Profesores: Pablo Badilla y Diego Cortez
- Auxiliares: Valentina Rojas y Melanie Peña
- Ayudantes: Javiera Arévalo, Tamara Carrasco e Ignacio Reyes

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Patricio Espinoza Acuña.
- Nombre de alumno 2: Javiera Romero Orrego.

---

### Reglas

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibido copiar.
- Uso de LLM (Copilot, Claude, Cursor, etc.) restringido a consultas, documentación y corrección de errores.

# Temas a tratar

- Lectura eficiente de datos en formato Parquet.
- Optimización del uso de memoria mediante conversión de tipos de datos.
- Paralelización de operaciones I/O con `ThreadPoolExecutor`.
- Comparación de implementaciones de predicción: Python, NumPy, Numba, pandas y Polars.
- Entrenamiento de modelos con RandomForestRegressor y efecto de `n_jobs`.
- Orquestación de pipelines de datos con Apache Airflow.

# Objetivos principales del laboratorio

- Cargar datos de canciones de Spotify desde archivos Parquet y optimizar su representación en memoria.
- Comparar el tiempo de lectura de archivos en serie vs. en paralelo.
- Analizar el impacto de distintas implementaciones (Python puro, NumPy, Numba, pandas, Polars) en el tiempo de predicción de un modelo lineal.
- Entrenar un RandomForestRegressor que prediga la valencia de canciones, comparando el efecto de la paralelización del entrenamiento.
- Orquestar el pipeline completo (carga + entrenamiento) usando Apache Airflow.

> Instalamos e importamos las librerías necesarias 🎸

In [1]:
!uv pip install pandas pyarrow lightgbm scikit-learn plotly apache-airflow polars numba

Using Python 3.14.3 environment at: c:\Users\HP\Desktop\MDS7202\.venv
Audited 8 packages in 209ms


In [2]:
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from pathlib import Path

import numba
import numpy as np
import pandas as pd
import plotly.express as px
import polars as pl
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

DATA_DIR = Path("data")

# 1. Carga y Optimización de Datos con Parquet

Los datos que usaremos en este laboratorio corresponden a un dataset de canciones de Spotify almacenado en **20 archivos Parquet** (`batch_01.parquet` … `batch_20.parquet`), con un total de 200 000 canciones y 24 columnas que incluyen características de audio, metadatos y la letra completa de cada canción.

A continuación trabajaremos en dos aspectos fundamentales de la carga de datos en la práctica:
1. **Optimizar el uso de memoria** ajustando los tipos de datos de las columnas.
2. **Reducir el tiempo de carga** paralelizando la lectura de archivos.

## 1.1 Exploración y Optimización de Tipos de Datos [1 Punto]

Cuando cargamos datos con pandas, los tipos inferidos por defecto no siempre son los más eficientes. Por ejemplo, un entero que siempre cabe en 16 bits se almacena por defecto como `int64` (64 bits), usando 4 veces más memoria de la necesaria. Lo mismo ocurre con flotantes y con columnas categóricas almacenadas como strings.

**Código dado** — funciones de carga:

In [3]:
def load_batch(path: str) -> pd.DataFrame:
    """Lee un único archivo Parquet y retorna un DataFrame."""
    return pd.read_parquet(path)


def load_all_serial(data_dir: Path, n_batches: int | None = None) -> pd.DataFrame:
    """Lee todos los archivos Parquet de data_dir en serie y los concatena."""
    paths = sorted(data_dir.glob("*.parquet"))
    if n_batches is not None:
        paths = paths[:n_batches]
    return pd.concat([load_batch(str(p)) for p in paths], ignore_index=True)

**TO-DO [0.3 Puntos]:**
- [ ] Ejecutar `load_all_serial` sobre todos los batches y explorar el DataFrame resultante (`.dtypes`, `.memory_usage(deep=True)`).
- [ ] Aplicar las siguientes conversiones a un nuevo DataFrame (copia del originalmente cargado `df_opt`):
  - `float64` → `float32`: columnas de audio features (`danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo`, `avg_artist_popularity`).
  - `int64` → `int16`: columnas `key`, `mode`.
  - `int64` → `int32`: columnas `year`, `popularity`, `duration_ms`, `total_artist_followers`.
- [ ] Comparar el uso de memoria antes y después con un gráfico de barras usando Plotly (código dado).

In [4]:
# Escribe aquí tu código
df = load_all_serial(DATA_DIR, n_batches=20)  # cargar todos los batches

print("Tipos de datos en el DataFrame original:")
print(df.dtypes)
print(f"Uso de memoria original: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MiB")

df_opt = df.copy()  # .astype(...)

# Aplicamos los cambios a la copia del df
# Convertir de float64 a float32 columnas "danceability", "energy", "loudness", "speechiness", "acousticness", "instrumentalness", "liveness", "valence", "tempo", "avg_artist_popularity"
float_cols = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "avg_artist_popularity",
]
df_opt[float_cols] = df_opt[float_cols].astype(np.float32)

# Convertir de int64 a int16 columnas "key" y "mode"
int_cols = ["key", "mode"]
df_opt[int_cols] = df_opt[int_cols].astype(np.int16)

# Convertir de int64 a int32 columnas "year", "popularity", "duration_ms", "total_artist_followers"
int_cols_32 = ["year", "popularity", "duration_ms", "total_artist_followers"]
df_opt[int_cols_32] = df_opt[int_cols_32].astype(np.int32)


# Compara el uso de memoria antes y después con un gráfico de barras
mem_before = df.memory_usage(deep=True).sum() / 1024**2
mem_after = df_opt.memory_usage(deep=True).sum() / 1024**2

px.bar(
    x=["Antes", "Después"],
    y=[mem_before, mem_after],
    labels={"x": "Estado", "y": "Uso de Memoria (MiB)"},
    title=f"Uso de memoria: {mem_before:.1f} MiB → {mem_after:.1f} MiB ({(1 - mem_after / mem_before) * 100:.1f}% reducción)",
).show()

Tipos de datos en el DataFrame original:
id                            str
name                          str
album_name                    str
artists                    object
danceability              float64
energy                    float64
key                         int64
loudness                  float64
mode                        int64
speechiness               float64
acousticness              float64
instrumentalness          float64
liveness                  float64
valence                   float64
tempo                     float64
duration_ms                 int64
lyrics                        str
year                        int64
genre                         str
popularity                  int64
total_artist_followers      int64
avg_artist_popularity     float64
artist_ids                 object
niche_genres               object
dtype: object
Uso de memoria original: 358.7 MiB


### Preguntas [0.7 Puntos]

1. ¿Qué es el formato **Parquet**? ¿Qué ventajas tiene sobre CSV para datos analíticos? ¿Qué es *columnar storage* y por qué acelera las consultas que solo leen algunas columnas?

Parquet es un formato de almacenamiento de archivos enfocado en cantidades grandes de información, por esto mismo su formato es binario y columnar a diferencia de csv. Sus principales ventajas con respecto a CSV para datos analíticos es el tiempo de carga de los datos, lo cual se debe a que en csv cada valor se trata como texto plano y luego se convierte en su tipo correcto, mientras que parquet es binario y los valores ya están en su correcta representación. Asimismo, la compresión nativa del formato parquet permite que ocupen menos bytes en disco y por tanto la lectura de esto sea más rápida, otra ventaja corresponde a que optimiza el tiempo de consulta gracias a su formato columnar, cuando se hace una consulta solo se leen las columnas relevantes, por último cuenta con una compresión más eficiente gracias al uso de binario en comparación a CSV que los trata como strings. El columnar storage se refiere a que en vez de guardar la información por filas como lo hace CSV, este lo hace por columna, por ejemplo si en csv tenemos info de tipo:  

Fila | Nombre | Edad

1    | Roku   | 4  
2    | Pato   | 24

en parquet tendremos algo como 


Columna Edad: [4, 24]
Columna Nombre: [Roku, Pato]


Esto acelera las consultas ya que cuando dicha consulta no requiera información de todas las columnas existentes, entonces podremos consultar solo las columnas de interés, ahorrandonos lecturas de más. 

2. ¿Qué es **Apache Arrow**? ¿Cómo se relaciona con Parquet y con pandas internamente? ¿Qué ganas al usar `pd.read_parquet` en vez de `pd.read_csv`?

Apache Arrow es una plataforme de desarrollo de software que define un formato estandarizado columnar y en memoria de los datos, esto permite que varios lenguajes o plataformas puedan utilizar los mismos datos sin tener que hacer transformaciones en ellos al moverse de uno a otro. Su relación con Parquet es que este nos permite optimizar el uso de datos en disco, mientras que Apache Arrow nos ayuda en memoria, por lo que idealmente para la mayor optimización los deberíamos ocupar juntos, con pandas se relaciona a través de que las versiones recientes de pandas utilizan Apache Arrow como motor interno para los datos. 

Al usar pd.read_parquet en vez de pd.read_csv ganamos todas las ventajas de eficiencia que hemos hablado gracias a Apache Arrow y el formato parquet, es decir, tendremos más velocidad en el procesamiento de los datos (lecturas, consultas, entre otros), optimizaremos el uso de memoria y además tendremos la preservación del tipo de dato, no tendremos que volver a definir qué es un float, int, string, etc. Por otro lado como CSV son strings pandas debería interpretar cada valor al momento de la carga. 


3. ¿Por qué existe `float32` si `float64` es más preciso? ¿En qué contextos esa pérdida de precisión es irrelevante?

La diferencia entre float32 y float64 es el numero de bytes que utilizan (4 y 8 respectivamente). Si bien esto permite que float64 sea más preciso, tambien significa que para grandes volumenes de datos la memoria ocupada se podría duplicar. Asimismo, esta ventaja en la precisión de los datos a costa de mayor memoria podria ser irrelevante en contexto de ML donde los modelos no requieren ese grado de precisión y existen otros errores que influyen más (como ruido de los datos), en este caso, el tiempo de entrenamiento o procesamiento se podría extender debido al uso de float64 y no notar una mejora. En otro contexto como DL, puede ocurrir que la pérdida de precisión sea irrelevante si usar float32 permite un mejor uso de los recursos de GPUs o librerías por sus optimizaciones nativas.


4. ¿Cuándo **no** conviene reducir la precisión de un tipo numérico? ¿Qué riesgos concretos existen?

No conviene reducir la precisión cuando se requiere alta exactitud numérica, como en coordenadas GPS o cálculos matemáticos avanzados, donde el error de redondeo de float32 se va acumulando durante las operaciones y puede producir resultados incorrectos.

Los riesgos concretos son la posibilidad de overflow (el valor supera el rango máximo representable, produciendo inf en float o un valor incorrecto en int) y  underflow (un valor demasiado pequeño se redondea a 0), así como la anterior mencionada propagación de error de redondeo en operaciones encadenadas, lo que nuestro contexto puede afectar el aprendizaje de un modelo si se usan tipos de baja precisión para los gradientes.


5. ¿Existe alguna alternativa a pandas para trabajar con estos datos de forma más eficiente en memoria? (menciona al menos dos)

Existen varias alternativas a pandas, en particular para un mejor uso de la memoria de podria usar Polars (construido sobre Apache Arrow) el cual soporta tanto float32 como float64 y sobre todo, realiza un paralelismo automático por columna a través de su evaluación lazy que evita materializar resutados intermedios.

Otra opción es Dask, el cual opera mediante una metodología de chunks sobre el dataset, esto permite que no se cargue todo directamente en la RAM de una sola vez. Al igual que Polars, permite paralelizar sobre múltiples cores.

6. ¿Cuánto se redujo el uso de memoria en total (en MiB y en %)? ¿Era esperable ese resultado? ¿Por qué no se redujo tanto como podría esperarse?
El uso de memoria en total se redujo de 358.7MiB a 345.7MiB, es decir en 13MiB o un 3,6%, era esperable que se redujiera el uso dado que cortamos a la mitad o más de su uso original multiples variables (pasando de 64 a 32 o 64 a 16). La reducción no fue tanta como hubieramos esperado ya que el dataset también cuenta con columnas de tipo str y object que no fueron intervenidas y puede que estas sean las variables que están ocupando la mayoría de la memoria total, sobre todo considerando que tenemos variables como "lyrics" que tienden a ser largas.

7. ¿Qué pasaría si intentaras reducir `valence` a `float16`? ¿Qué riesgo existiría para el modelo entrenado en la sección 2?
Si se reduce valence a float16 se tendrán aún menos digitos significativos disponibles para esta variable, puede ocurrir que canciones que antes llegaban a tener una ligera diferencia ahora se asemejen más (y aunque no tan común, queden con una misma valence).

El riesgo principal para el modelo entrenado en la parte 2 es que la variable a predecir es valence, y por tanto, los datos con un float16 tendrán una cantidad importante de ruido agregado debido a los redondeos realizados. Otra posibilidad sería que un modelo entrenado con labels ruidosos (float16) aprenderá a predecir valores artificialmente discretizados y en un entorno de producción, sus predicciones tendrán un error sistemático sobre datos con otro tipo de float superior. Si bien en la sección 2 se utiliza una regresión, los fallos que puede generar esta transformación tampoco permitirán identificar claramente si el problema es el modelo, o si ocurre debido a la conversión de la variable.


**Escribe tus respuestas aquí...**

In [5]:
# **IMPORTANTE**: Una vez contestada la pregunta, ejecutar esta celda para liberar memoria.
df_opt = None

## 1.2 Lectura en Serie vs. Paralelo [1 Punto]

Cuando se trabaja con múltiples archivos, la lectura **en paralelo** puede reducir el tiempo total al aprovechar que la espera de I/O (disco/red) no bloquea al procesador. En Python, la clase `ThreadPoolExecutor` del módulo `concurrent.futures` permite lanzar múltiples hilos para ejecutar operaciones de forma concurrente.

**TO-DO: [0.3 Puntos]**
- [ ] Implementar `load_all_parallel` usando `ThreadPoolExecutor`.
- [ ] Medir con `%timeit` ambas versiones sobre todos los batches.
- [ ] Generar un gráfico de línea (Plotly) con los tiempos para 2, 4, 6, …, 20 archivos, con series `Serial` y `Paralelo`.

In [6]:
# Escribe aquí tu código

# Implementamos load_all_parallel usando ThreadPoolExecutor para cargar los batches en paralelo
def load_all_parallel(data_dir: Path, n_batches: int | None = None) -> pd.DataFrame:
    """Lee todos los archivos Parquet de data_dir en paralelo y los concatena."""
    paths = sorted(data_dir.glob("*.parquet"))
    if n_batches is not None:
        paths = paths[:n_batches]

    with ThreadPoolExecutor() as executor:
        dfs = list(executor.map(lambda p: load_batch(str(p)), paths))

    return pd.concat(dfs, ignore_index=True)

**Benchmark:** mide tiempos para 2, 4, 6, ..., 20 archivos y grafica


In [7]:
@dataclass
class ReadMeasurement:
    n_files: int
    time_sec: float
    version: str


measurements: list[ReadMeasurement] = []

for n in range(2, 21):
    t0 = time.perf_counter()
    load_all_serial(DATA_DIR, n_batches=n)
    measurements.append(ReadMeasurement(n, time.perf_counter() - t0, "Serial"))

    t0 = time.perf_counter()
    load_all_parallel(DATA_DIR, n_batches=n)
    measurements.append(ReadMeasurement(n, time.perf_counter() - t0, "Paralelo"))

df_times = pd.DataFrame(measurements)
px.line(
    df_times,
    x="n_files",
    y="time_sec",
    color="version",
    markers=True,
    title="Tiempo de lectura: Serial vs Paralelo",
    labels={"n_files": "Número de archivos", "time_sec": "Tiempo (s)"},
).show()

### Preguntas  [0.7 Puntos]

1. ¿Qué significa que una operación sea **I/O-bound** vs **CPU-bound**? ¿A cuál categoría pertenece la lectura de archivos desde disco?

Una operación I/O bound son operaciones donde el tiempo de ejecucción viene dado por las operaciones de entrada y salida (Input/Output por eso I/O), es decir operaciones como leer archivos o consultar a una base de datos, en estos casos el procesador pasa la gran parte del tiempo simplemente esperando que lleguen los datos solicitados. Por otro lado,CPU-bound son operaciones donde ahora el tiempo de ejecución viene dado por cálculos hechos en el mismo procesador, por ejemplo entrenamientos de modelos, por ende estas operaciones vienen limitadas por la capacidad propia del procesador. 

La lectura de archivos desde disco, corresponde a una operación I/O bound pues la respuesta viene limitada por la capacidad de acceso al almacenamiento (disco) y no la capacidad de la CPU.

2. ¿Qué es el **GIL** (*Global Interpreter Lock*) de CPython? ¿Por qué existe? ¿Qué problema resuelve y qué limitación introduce?
Es un mecanismo usado en el interpretador que asegura que solo un hilo se ejecute dentro de un mismo proceso, existe para resolver problemas de manejo de memoria, provocados por ejemplo por memory leaks al tener multiples punteros de distintos punteros a una misma referencia y también existe para proteger estructuras internas de datos, gracias a la simplicidad de tener un solo hilo donde se maneje la memoria. La limitación que introduce tiene que ver con las tareas CPU-bound, estas demorarán más cuando son pesadas pues no permitirá que se usen multiples cores para paralelizar el proceso.


3. ¿Por qué usamos Python si tiene el GIL? ¿Qué ganamos al usarlo como lenguaje de *pegamento* entre librerías de alto rendimiento (NumPy, Arrow, PyTorch…)?

Usamos Python exactamente por el hecho de que es un buen "lenguaje de pegamento" con esto nos referimos a que utilizando python podemos mezclar varias herramientas de alto rendimiento como pandas, numpy, scikit-learn, etc. Python solamente las "coordina" con una sintáxis más amigable y luego le da el control a la librería, así no necesitamos escribir código en lenguajes como c/c++ pero aprovechamos igualmente su eficiencia, las librerías sí pueden aprovechar por ejemplo el uso de múltiples threads gracias a eso, al compilarse en c/c++ en vez del propio python. 

4. ¿Cuándo conviene usar `ThreadPoolExecutor` vs `ProcessPoolExecutor`? ¿Cuál usarías si la operación fuera puramente CPU-bound?

Conviene usar ThreadPoolExecutor cuando las tareas son del tipo I/O-bound ya que los hilos compartirán memoria y tendrán un bajo overhead, además que como el cuello de botella ocurre por la espera de los datos externos, entonces el GIL no representará un problema.

ProcessPoolExecutor es conveniente de usar para tareas del tipo CPU-bound ya que crea procesos independientes entre sí con su proprio interprete de python, esto provoca una especie de bypass al GIL, lo que genera el paralelismo deseado.

Para una operación pura de CPU-bound conviene usar ProcessPoolExecutor para maximizar los recursos disponibles en los núcleos del procesador.

5. ¿Qué overhead introduce crear un pool de threads? ¿Qué pasaría si los archivos fueran muy pequeños (p.ej. 1 KB cada uno)?

La tarea de crear un pool de threads tiene un overhead fijo que consiste en la inicialización de los threads, la mantenciónb de las colas, realizar context switch entre hilos y finalmente sincronizar los resultados de estos. 

Si los archivos fueran muy pequeños, entonces podría ocurrir que el overhead sea superior al tiempo real de la lectura, y por tanto, se gastaría más tiempo en administrar los hilos (overhead) que en leer los datos realmente. Este caso representaría un ejemplo de que la lectura paralela es más lenta que la secuencial, y que por tanto no siempre es la mejor opción o un 'must-do'.

6. ¿Se observó mejora con la lectura paralela? ¿A partir de cuántos archivos empieza a ser notable?

Si se observo mejora con la lectura paralela, en el caso de 20 archivos el tiempo en paralelo es muy cercano a la mitad del secuencial.

A partir de los 8 archivos se puede notar la diferencia entre ambos métodos, y esta diferencia aumenta cada vez más a medida que se agregan más archivos.

7. ¿Por qué el speedup obtenido **no es igual** al número de threads disponibles? ¿Qué factores lo limitan?

El speedup (tiempo secuencial/tiempo paralelo, ej: x2, x4) no es igual a la cantidad de threads disponibles ya que hay procesos como inicializar el pooling o combinar los resultados que no se paraleliza. Asimismo, el context switching tambien genera overhead puesto que hay un costo asociado a que el sistema operativo alterne entre los hilos.
Finalmente, también hay un limite propio del almacenamiento en el cual más threads no aumentarán la velocidad del disco si este ya se encuentra en su limite (saturación del ancho de banda).


**Escribe tus respuestas aquí...**

# 2. Predicción de Valencia

La columna `valence` de Spotify mide el **positivismo musical** de una canción: valores cercanos a 1 indican canciones alegres y eufóricas, mientras que valores cercanos a 0 corresponden a canciones tristes o melancólicas. En esta sección analizaremos distintas formas de realizar predicciones con un modelo de regresión lineal ya entrenado, y luego entrenaremos un modelo más complejo.

## 2.1 Regresión Lineal a Mano [1.5 Puntos]

Antes de entrenar un modelo completo, veremos cómo **la elección de implementación** afecta drásticamente el rendimiento de predicción. Usaremos un modelo de regresión lineal pre-entrenado cuyos coeficientes ya están dados, e implementaremos la predicción usando cinco enfoques distintos: Python puro, NumPy, Numba (JIT), pandas y Polars.

**Código dado — carga de datos y parámetros del modelo:**

In [8]:
# Carga de datos y preparación del split
df_train = load_all_serial(DATA_DIR, n_batches=20)

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]

X = df_train[PARAM_COLS + ["key", "mode", "genre"]]
y = df_train["valence"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [9]:
# Parámetros del modelo lineal pre-entrenado (dados)
params = {
    "danceability": 0.7718203106411208,
    "energy": 0.4252134896942928,
    "loudness": -0.008319917439312445,
    "speechiness": -0.24543273088867107,
    "acousticness": 0.10440236785191129,
    "instrumentalness": -0.11203723673701874,
    "liveness": 0.023790522969698424,
    "tempo": 0.0007885690378158087,
    "duration_ms": -4.31739613602265e-07,
    "year": -0.0036043842721972985,
}
intercept = 6.948154825159983

params_vals = list(params.values())
params_arr = np.array(params_vals, dtype=np.float32)

**Código dado — las 5 implementaciones de predicción:**

Analiza cómo cada implementación aborda el mismo problema y presta atención a las diferencias en legibilidad, concisión y (como verás en el benchmark) rendimiento.

In [10]:
def linear_regression_predict(X: np.ndarray, params: list[float]) -> list[float]:
    """Predicción con loop Python puro."""
    preds = []
    for row in X:
        val = intercept
        for j, w in enumerate(params):
            val += row[j] * w
        preds.append(val)
    return preds


def linear_regression_predict_numpy(X: np.ndarray, params: list[float]) -> np.ndarray:
    """Predicción vectorizada con NumPy."""
    return (X * np.array(params)).sum(axis=1) + intercept


@numba.njit
def linear_regression_predict_numba(X: np.ndarray, params: np.ndarray) -> np.ndarray:
    """Predicción con Numba JIT (loop compilado a código máquina)."""
    n = X.shape[0]
    preds = np.empty(n)
    for i in range(n):
        val = intercept
        for j in range(len(params)):
            val += X[i, j] * params[j]
        preds[i] = val
    return preds


def linear_regression_predict_pandas(X: pd.DataFrame, params: list[float]) -> pd.Series:
    """Predicción vectorizada con pandas (dot product)."""
    return X.dot(pd.Series(params, index=X.columns)) + intercept


def linear_regression_predict_polars(X: pl.DataFrame, params: list[float]) -> pl.Series:
    """Predicción vectorizada con Polars (expresiones lazy)."""
    weights = dict(zip(X.columns, params, strict=False))
    expr = pl.lit(intercept)
    for col, w in weights.items():
        expr = expr + pl.col(col) * w
    return X.select(expr.alias("pred"))["pred"]

**Código dado — Benchmark de las 5 implementaciones:**

In [11]:
@dataclass
class TimeMeasurement:
    time_took: float
    iteration: int
    version: str


time_measurements: list[TimeMeasurement] = []

ranges = [10, 50, 100, 250, 500, 750, 1000, *range(1001, len(X_test) + 1, 1000)]

for it in ranges:
    X_np = X_test[PARAM_COLS].iloc[:it].to_numpy(dtype=np.float32)
    X_pd = X_test[PARAM_COLS].iloc[:it]
    X_pl = pl.from_pandas(X_pd)

    for name, fn, args in [
        ("Python", linear_regression_predict, (X_np, params_vals)),
        ("NumPy", linear_regression_predict_numpy, (X_np, params_vals)),
        ("Numba-JIT", linear_regression_predict_numba, (X_np, params_arr)),
        ("Pandas", linear_regression_predict_pandas, (X_pd, params_vals)),
        ("Polars", linear_regression_predict_polars, (X_pl, params_vals)),
    ]:
        t0 = time.perf_counter()
        fn(*args)
        time_measurements.append(TimeMeasurement(time.perf_counter() - t0, it, name))

df_bench = pd.DataFrame(time_measurements)

# Gráfico 1: tiempos absolutos
px.line(
    df_bench,
    x="iteration",
    y="time_took",
    color="version",
    markers=True,
    title="Tiempos de predicción según implementación",
    labels={"iteration": "Número de filas", "time_took": "Tiempo (s)"},
).show()

# Gráfico 2: tiempos absolutos (en log)
px.line(
    df_bench,
    x="iteration",
    y="time_took",
    color="version",
    markers=True,
    title="Tiempos de predicción según implementación (en escala logarítmica)",
    labels={"iteration": "Número de filas", "time_took": "Tiempo (s)"},
    log_y=True,
).show()

# Gráfico 3: speedup relativo respecto a Python puro
pivot = df_bench.pivot(index="iteration", columns="version", values="time_took")
for col in ["NumPy", "Numba-JIT", "Pandas", "Polars"]:
    pivot[col] = pivot["Python"] / pivot[col]
pivot["Python"] = 1.0

melted = pivot.reset_index().melt(
    id_vars=["iteration"],
    value_vars=["Python", "NumPy", "Numba-JIT", "Pandas", "Polars"],
    value_name="speedup",
)
px.line(
    melted,
    x="iteration",
    y="speedup",
    color="version",
    markers=True,
    title="Speedup relativo respecto a Python puro",
    labels={"iteration": "Número de filas", "speedup": "Speedup (×)"},
).show()

### Preguntas [1.5 Puntos]

  1. ¿Qué es la vectorización en NumPy? ¿Cómo puede ejecutar operaciones sobre arrays sin loops de Python explícitos?

  La vectorización son operaciones aplicadas sobre arrays completos en lugar de hacer la iteración individual de elemento por elemento mediante loops de Python explicitos. En especifico, estas operaciones son realizadas en C por numpy, la cual itera sobre los elementos pero sin usar el interprete de Python, lo que ayuda a eliminar el overhead del interprete por cada elemento, permitiendo a su vez que el compilador aplique optimizaciones adicionales en CPU. 

  2. ¿Qué es JIT (Just-In-Time compilation)? ¿Qué hace el decorador @numba.njit? ¿Qué significa el modo nopython?

  El JIT es una ténica que complia código en tiempo de ejecución justo previo a su ejecución, evitando así compilarlo de forma anticipada.

  El decorador @numba.njit indica a numba que compile una función a código máquina usando Low Level Virtual Machine cuando sea llamda por primera vez. LLVM funciona como una infraestructura de compilación que toma codigo intermedio y lo convierte en código de maquina optimizado para el procesador en donde corre.

  El modo nopython, que es activado mediante njit, implica que Numba debe compilar la función sin recurrir al interprete Python, para ello todas las variables deben de tener tipos inferibles y las operaciones permitidas son solo aquellas soportadas por Numba. Si bien esto genera más restricciones, la ventaja es que el código será más rápido de ejecutar.

  3. ¿Por qué Numba es más lento en la primera ejecución? ¿Qué es el warm-up de JIT y cómo lo manejamos en el benchmark?

  Numba es más lento en la primera ejecución ya que no tiene un código de maquina precompilado, y por tanto, debe de realizar todo el proceso de: analizar la función, inferir tipos de los argumentos, crear el codigo LLVM IR y posteriormente compilarlo a instrucciones para la máquina. 

  El warm-up de jit corresponde al coste inicial de todo el proceso y tareas asociadas mencionado anteriormente. En el benchmark el warm-up ocurre en la primera iteración, donde Numba aparece como el más lento. Por otra parte, las iteraciones siguientes ya usarán el código precompilado, siendo más rápidas. Dentro del benchmark, si no se desea considerar este costo en los tiempos, entonces se debe realizar una primera llamada a modo de warm-up previo a las mediciones.
  
  4. ¿Qué es Polars y cuáles son sus principales características como librería de datos? ¿Para qué escenarios fue diseñada y por qué ha ganado popularidad como alternativa a pandas?

  Polars es un libreria orientada a dataframes la cual fue escrita en Rust sobre Apacha Arrow. Sus principales características son la evaluación lazy con optimización de queries, paralelismo automatico y manejo de memoria eficiente mediante Arrow (zero-copy). 

  Fue diseñada para datasets grandes que caben en memoria, de forma que se pueda priorizar el rendimiento y consistencia. Su popularidad ha aumentado como alternativa a pandas debido a que es más rápida en operaciones como groupby o joins, consumiendo menos memoria.
  
  5. ¿En qué se diferencia Polars de pandas a nivel de implementación (lenguaje, modelo de ejecución, manejo de memoria)?

  - Polaris usa como lenguaje base Rust mientras que pandas utiliza Python y C (para numpy).
  - Polaris usa como modelo de ejecución Lazy por defecto, mientras que pandas utiliza Eager, en donde cada operación se ejecuta inmediatamente.
  - Polaris maneja la memoria mediante Apacha Arrow por columna y zero-copy, mientras que pandas utiliza los Arrays de numpy.

  6. ¿Por qué pandas puede ser más lento que NumPy aun usando operaciones vectorizadas internamente?

  Pese a que pandas utiliza numpy, de forma interna tambien tiene procesos que generan un overhead adicional como lo pueden ser las verificaciones y alineaciones dede los indices previo a operar, información adicional de metadata la cual se mantiene en las Series y Dataframe y también el manejo de tipos especiales como lo son los null, los que a su vez requieren verificaciones extra para las operaciones.
  
  7. ¿Qué son las instrucciones SIMD (Single Instruction Multiple Data)? ¿Cómo contribuyen a la aceleración de NumPy y Polars?

  Las instrucciones SIMD son propias a la CPU y se encargan de que una operación sea aplicada a multiples datos en paralelo dentro de lo que es un mismo ciclo de reloj.

  Estas instrucciones aceleran Numpy ya que operaciones como la suma o multiplicación de los arrays requieren procesar varios elementos por ciclo. A su vez, Polars también se beneficia de SIMD por esta misma razón. Asimismo, debido a que ambos tienen soporte para este tipo de instrucciones SIMD, ambos resultan tener ventaja sobre los loops puros de Python.
  
  8. ¿Cuándo conviene usar Numba sobre NumPy? ¿Y Polars sobre pandas para operaciones numéricas?


  
  9. ¿Cuál implementación fue la más rápida en tu medición? ¿Era esperable ese resultado?


  
  10. ¿Se observa diferencia notable entre pandas y NumPy? ¿Por qué pandas puede ser más lento o más rápido?


  
  11. ¿A partir de cuántas filas empieza a ser evidente la ventaja de NumPy/Numba sobre Python puro?


  
  12. ¿Polars fue más eficiente que pandas en tu medición? Verifica la versión de pandas instalada (pd.__version__) y comenta si crees que la versión influye en el resultado.


  
  13. ¿Por qué Numba puede igualar o superar a NumPy para loops numéricos simples?


  
  14. El benchmark excluye el costo de convertir datos a NumPy/Polars (la conversión ocurre fuera del timing). ¿Cómo cambiaría el resultado si incluyeras ese costo? ¿En qué escenarios de producción ese costo no existiría?


  
  15.  Si tuvieras que realizar esta predicción sobre 100 millones de filas en un servidor de producción, ¿qué implementación elegirías y por qué? ¿Cambiaría tu respuesta si dispusieras de una GPU?


  

**Escribe tus respuestas aquí...**

### 2.2 Entrenamiento y Comparación de `n_jobs` [0.5 Puntos]

Ahora entrenaremos un modelo más complejo: un **RandomForestRegressor** que usa las características de audio más una codificación del género musical para predecir `valence`. Compararemos el efecto de paralelizar el entrenamiento con el parámetro `n_jobs`.

**Código dado — pipeline encapsulado** (no modificar):

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


def build_pipeline(n_jobs: int = 1) -> Pipeline:
    # En producción este pipeline usaría LGBMRegressor; aquí usamos RandomForest
    # para ilustrar el efecto de n_jobs de forma más pronunciada.
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        (
                            "numerical",
                            "passthrough",
                            PARAM_COLS,
                        ),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )

In [13]:
# Entrena con n_jobs=1 y mide el tiempo
pipeline_1 = build_pipeline(n_jobs=1)
t0 = time.perf_counter()
pipeline_1.fit(X_train, y_train)
time_1job = time.perf_counter() - t0

# Entrena con n_jobs=-1 y mide el tiempo
pipeline_all = build_pipeline(n_jobs=-1)
t0 = time.perf_counter()
pipeline_all.fit(X_train, y_train)
time_all_jobs = time.perf_counter() - t0

# Calcula RMSE de ambos modelos
rmse_1 = root_mean_squared_error(y_test, pipeline_1.predict(X_test))
rmse_all = root_mean_squared_error(y_test, pipeline_all.predict(X_test))

print(f"n_jobs=1  → tiempo: {time_1job:.1f}s | RMSE: {rmse_1:.4f}")
print(f"n_jobs=-1 → tiempo: {time_all_jobs:.1f}s | RMSE: {rmse_all:.4f}")

n_jobs=1  → tiempo: 364.6s | RMSE: 0.1652
n_jobs=-1 → tiempo: 45.7s | RMSE: 0.1652


In [14]:
# Gráficos de tiempos y RMSE
df_perf = pd.DataFrame(
    {
        "configuracion": ["n_jobs=1", "n_jobs=-1"],
        "tiempo_s": [time_1job, time_all_jobs],
        "rmse": [rmse_1, rmse_all],
    }
)

px.bar(
    df_perf,
    x="configuracion",
    y="tiempo_s",
    title="Tiempo de entrenamiento según n_jobs",
    labels={"tiempo_s": "Tiempo (s)", "configuracion": "Configuración"},
    text_auto=".1f",
).show()

px.bar(
    df_perf,
    x="configuracion",
    y="rmse",
    title="RMSE según n_jobs",
    labels={"rmse": "RMSE", "configuracion": "Configuración"},
    text_auto=".4f",
).show()

### Preguntas [0.5 Puntos]

1. ¿Qué hace el parámetro `n_jobs` en RandomForest (y en general en scikit-learn)?

El parámetro en n_jobs se encarga de controlar cuantos trabajos paralelos usará el algoritmo. En algoritmos como RandomForest, en donde los arboles son independientes entre sí, se puede realizar un entrenamiento simultaneo para optimizar el tiempo total de entrenamiento. En general, n_jobs=1 entrenará cada arbol uno por uno, n_jobs=-1 usará toda la CPU disponible para el entrenamiento de los árboles. En general, n_jobs permite definir el nivel de paralelización en aquellos procesos donde es posible, como gridsearch o validación cruzada.

2. **¿Por qué aquí sí funciona el paralelismo real sin el problema del GIL?** (Pista: 
RandomForest en scikit-learn usa joblib con backend de procesos o threads nativos.)

El paralelismo si funciona ya que RandomForest en scikit-learn usa joblib en el backend, esto crea procesos separados y no threads. Por tanto, cada proceso tendrá su propio interprete de Python con su propio GIL, lo que evita que un proceso bloquee a otro y por tanto que el paralelismo si funcione. 

3. ¿Cuánto mejoró el tiempo con `n_jobs=-1`? 

El tiempo al usar n_jobs=-1 se redujo de 364.6 segundos a 45.7 segundos, siendo un speddup de approx x7.98

4. ¿Fue proporcional al número de CPUs disponibles en tu máquina? ¿Por qué no?

La máquina usada para estos resultados tiene 8 núcleos, por tanto el speedup coincide con el número de CPUs disponibles. Sin embargo, esto no siempre ocurre debido a lo mencionado en preguntas anteriores, donde el overhead producido por inicialización de pipeline, creación y coordinación de los procesos en joblib, memorias compartidas y encolamiento del sistema operativo donde los recursos compiten contra otros procesos del sistema generan un coste adicional en tiempo, disminuyendo la proporcionalidad o speedup real versus el numero de núcleos usados.

5. ¿Hubo diferencia en RMSE entre ambas versiones? ¿Era esperable? ¿Por qué?

No hubo una diferencia en los RMSE de ambas versiones, ambos tuvieron 0.1652. Esto era esperable ya que n_jobs es un parametro que solo controla la distribución de la tarea, en este caso ambas versiones ejecutaron exactamente el mismo proceso, la diferencia fue la cantidad de recursos de CPU utilizados.


**Escribe tus respuestas aquí...**

# 3. Orquestación del Pipeline con Apache Airflow

En producción, los pipelines de datos y ML rara vez se ejecutan a mano desde un notebook. Se necesita:
- **Automatización**: que el pipeline corra periódicamente (diariamente, por hora…).
- **Dependencias**: que el entrenamiento solo comience si la carga de datos terminó exitosamente.
- **Monitoreo y reintentos**: que si una tarea falla, el sistema lo registre y reintente.

**Apache Airflow** resuelve exactamente esto. Define pipelines como **DAGs** (*Directed Acyclic Graphs*), donde cada nodo es una **tarea** y las aristas definen dependencias.

| Concepto | Descripción |
|----------|-------------|
| **DAG** | Grafo Dirigido Acíclico que representa el pipeline completo |
| **Operator** | Unidad de trabajo (`PythonOperator`, `BashOperator`, …) |
| **Task** | Instancia de un Operator dentro de un DAG |
| **XCom** | Mecanismo para pasar datos pequeños entre tareas |
| **schedule** | Expresión cron que indica cuándo ejecutar el DAG |

### Setup local


En la carpeta del Lab:

```bash
export AIRFLOW_HOME=$(pwd)
airflow db migrate          # inicializa la base de datos de metadata
# Ver la contraseña. Si no se en un comienzo, ejecutar airflow standalone, parar el proceso y luego ejecutar nuevamente este comando. 
cat $AIRFLOW_HOME/simple_auth_manager_passwords.json.generated 
airflow standalone       # levanta scheduler + webserver en http://localhost:8080
```

Los DAGs deben guardarse en `./dags`.

## 3.1 Implementación del DAG

**TO-DO [0.8 Puntos]:**
- [ ] Implementar `task_load_data_fn`: cargar 5 batches en paralelo, guardar en disco como Parquet y pasar la ruta a la siguiente tarea usando XCom.
- [ ] Implementar `task_train_model_fn`: recuperar la ruta de XCom, cargar el DataFrame, preparar X e y, entrenar `build_pipeline(n_jobs=-1)` e imprimir el tiempo.
- [ ] Definir la dependencia entre tareas (`load_data >> train_model`).

El siguiente bloque es el template que debes completar en tu celda de respuesta.

In [15]:
%%writefile ~/airflow/dags/spotify_pipeline_dag.py

from pathlib import Path
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split


DATA_DIR = Path("/RUTA/ABSOLUTA/A/Labs/Lab9_v2/data")  # AJUSTA esta ruta
OUTPUT_PATH = Path("/tmp/spotify_data.parquet")

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]


# ── Funciones auxiliares (dadas) ─────────────────────────────────────────────


def load_batch(path: str) -> pd.DataFrame:
    return pd.read_parquet(path)


def load_all_parallel(data_dir: Path, n_batches: int = 5) -> pd.DataFrame:
    paths = sorted(data_dir.glob("*.parquet"))[:n_batches]
    with ThreadPoolExecutor(max_workers=None) as executor:
        dfs = list(executor.map(load_batch, [str(p) for p in paths]))
    return pd.concat(dfs, ignore_index=True)


def build_pipeline(n_jobs: int = -1) -> Pipeline:
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        ("numerical", "passthrough", PARAM_COLS),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )


# ── Funciones de las tareas de Airflow ───────────────────────────────────────


def task_load_data_fn(**context):
    """
    Carga 5 batches de datos en paralelo y guarda el resultado en disco.
    TODO: implementa esta función.
    - Usa load_all_parallel para cargar los datos.
    - Guarda el DataFrame resultante en OUTPUT_PATH (formato parquet).
    - Usa XCom para pasar la ruta del archivo a la siguiente tarea.
    """
    ...


def task_train_model_fn(**context):
    """
    Carga los datos desde disco y entrena el pipeline.
    TODO: implementa esta función.
    - Recupera la ruta del archivo desde XCom.
    - Lee el DataFrame desde esa ruta.
    - Prepara X e y, realiza el split 80/20.
    - Entrena build_pipeline(n_jobs=-1).
    - Imprime el tiempo de entrenamiento.
    """
    ...


# ── Definición del DAG ────────────────────────────────────────────────────────

with DAG(
    dag_id="spotify_pipeline",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["mds7202", "spotify"],
) as dag:
    load_data = PythonOperator(
        task_id="load_data",
        python_callable=task_load_data_fn,
    )

    train_model = PythonOperator(
        task_id="train_model",
        python_callable=task_train_model_fn,
    )

    # TODO: define la dependencia entre tareas (load_data debe ejecutarse antes que train_model)
    ...


Writing C:\Users\HP/airflow/dags/spotify_pipeline_dag.py


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\HP/airflow/dags/spotify_pipeline_dag.py'

In [ ]:
# Escribe aquí tu código (copia el template y completa los TODOs)


Una vez guardado el archivo, ejecuta el DAG con:

```bash
airflow standalone
```


### Pega aquí el output de las steps del DAG

- Step 1: 
...


- Step 2:
...

### Preguntas [1.2 Puntos]

1. ¿Qué es un **DAG**? ¿Qué significa que sea *Directed* (dirigido) y *Acyclic* (acíclico)? ¿Por qué importa la propiedad acíclica en un pipeline de datos?
2. ¿Qué es **Apache Airflow**? ¿Para qué tipo de problemas está diseñado y cuál es su unidad mínima de trabajo?
3. ¿Qué son los **Operators**? ¿Qué diferencia hay entre `PythonOperator` y `BashOperator`? ¿Cuándo usarías cada uno?
4. ¿Qué es **XCom** en Airflow? ¿Cómo funciona internamente (¿dónde se almacena?)? ¿Por qué **no** es adecuado para pasar DataFrames grandes entre tareas?
5. ¿Qué alternativa concreta usaste para pasar el DataFrame entre `load_data` y `train_model`? ¿Cuál sería la alternativa recomendada en producción (S3, GCS, DVC…)?
6. ¿Qué es el parámetro `schedule` de un DAG? ¿Cómo lo configurarías para que corra todos los días a las 3 AM?
7. ¿Qué diferencia hay entre Airflow y otras herramientas como **Prefect**, **Dagster**, **Luigi**, **Kubeflow**? ¿Cuál es la principal crítica que se le hace a Airflow?
8. ¿Por qué conviene orquestar el pipeline en Airflow en vez de simplemente ejecutar un script Python end-to-end?
9. ¿Qué pasa si `load_data` falla a mitad de camino? ¿Airflow reintenta automáticamente? ¿Cómo controlarías el número máximo de reintentos?
10. ¿Qué ventaja tiene que las tareas estén separadas (carga y entrenamiento) vs. una sola tarea monolítica, desde el punto de vista de debugging y eficiencia?
11. ¿Cómo podemos alertar si es que algún paso falla? ¿O si la pipeline se ejecuta correctamente?
12. En un pipeline de producción real, ¿qué otras tareas añadirías al DAG?

**Escribe tus respuestas aquí...**

# Conclusión

Eso ha sido todo para el lab de hoy. Recuerda que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda, no dudes en contactarnos por el foro de U-Cursos.

<p align="center">
  <img src="https://media.giphy.com/media/l0HlBO7eyXzSZkJri/giphy.gif" width="300">
</p>